# La performance en Python

Un code **correct** fait ce qu'on lui demande.
Un code **performant** le fait en utilisant le moins de **temps** et de **mémoire** possible.

### Les deux ressources à surveiller

| Ressource | Outil |
|---|---|
| ⏱ Temps CPU | `timeit` |
| 🧠 Mémoire RAM | `tracemalloc` |

> **Règle d'or : mesure d'abord, optimise ensuite.**
> Ne jamais supposer qu'un code est lent — le prouver avec des chiffres.

---
## 1. Mesurer le temps — `timeit`

`timeit.timeit()` exécute un bout de code **N fois** et retourne le **temps total** en secondes.
Répéter des milliers de fois élimine les variations aléatoires du système.

### Paramètres

| Paramètre | Description | Défaut |
|---|---|---|
| `stmt` | Le code à mesurer — chaîne ou lambda | obligatoire |
| `number` | Nombre de répétitions | `1_000_000` |
| `setup` | Code exécuté **une seule fois** avant les mesures | `'pass'` |

→ **Temps moyen** par exécution : `temps_total / number`

> **Forme recommandée** : lambda — elle a accès direct aux variables du notebook.
> La forme string (code en guillemets) tourne dans un espace isolé et ne voit pas tes variables.

In [ ]:
import timeit

# ── Forme lambda : accès direct aux variables du notebook ──
def additionner(liste):
    return sum(liste)

ma_liste = list(range(10_000))

temps = timeit.timeit(
    stmt=lambda: additionner(ma_liste),
    number=5_000          # répète 5000 fois
)

print(f"Temps total (5000 appels) : {temps:.4f}s")
print(f"Temps moyen par appel     : {temps / 5_000 * 1_000_000:.2f} µs")
# µs = microsecondes = millionièmes de seconde

Temps total (5000 appels) : 0.3034s
Temps moyen par appel     : 60.68 µs


In [ ]:
def quick_sort(liste : list):
    if len(liste) <= 1:
        return liste
    
    pivot = liste.pop()
    petit = []
    grand = []
    
    for v in liste:
        if v < pivot:
            petit.append(v)
        else:
            grand.append(v)
            
    return quick_sort(petit)+[pivot]+ quick_sort(grand)


values = [5,99,2,3,8,46,5,4,6,8,9]

temps = timeit.timeit(
    stmt=lambda: quick_sort(values),
    number=5_000          # répète 5000 fois
)

print(f"Temps total (5000 appels) : {temps:.4f}s")
print(f"Temps moyen par appel     : {temps / 5_000 * 1_000_000:.2f} µs")

Temps total (5000 appels) : 0.0036s
Temps moyen par appel     : 0.73 µs


In [ ]:
import timeit

# ── Comparer deux approches : += vs join() ──
# number=1000 : 1000 répétitions de chaque

code_lent = """
resultat = ""
for i in range(1000):
    resultat += str(i)
"""

code_rapide = """
resultat = "".join(str(i) for i in range(1000))
"""

t1 = timeit.timeit(code_lent,   number=1000)
t2 = timeit.timeit(code_rapide, number=1000)

print(f"+=     : {t1:.4f}s")
print(f"join() : {t2:.4f}s")
print(f"→ join() est {t1/t2:.1f}x plus rapide")

---
## 2. Mesurer la mémoire — `tracemalloc`

`tracemalloc` surveille la mémoire réservée par Python entre deux points du code.

### Utilisation standard

```python
tracemalloc.start()
# ... ton code ...
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
```

| Valeur | Description |
|---|---|
| `current` | Mémoire utilisée à cet instant |
| `peak` | **Maximum** atteint depuis le `start()` — c'est le chiffre important |

Les valeurs sont en **bytes**. Divise par `1_000_000` pour avoir des MB.

> `sys.getsizeof()` retourne la taille d'un objet, mais seulement la structure (le conteneur),
> pas la taille totale du contenu. Pour la vraie consommation mémoire, utilise `tracemalloc`.

In [ ]:
import tracemalloc

N = 500_000

# ── Version liste : tout en mémoire d'un coup ──
tracemalloc.start()
liste = [x**2 for x in range(N)]
_, pic_liste = tracemalloc.get_traced_memory()
tracemalloc.stop()

# ── Version générateur : rien en mémoire ──
tracemalloc.start()
gen = (x**2 for x in range(N))   # juste l'objet générateur, pas les valeurs
_, pic_gen = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Liste      : {pic_liste / 1_000_000:.1f} MB")
print(f"Générateur : {pic_gen} bytes")
print(f"→ Le générateur utilise environ {pic_liste // pic_gen}x moins de mémoire")

---
## 3. La complexité algorithmique — Big O

Big O mesure comment le temps d'exécution évolue quand les données **grandissent**.
C'est la question : *si je double mes données, qu'est-ce qui arrive à mon temps d'exécution ?*

| Notation | Nom | Si n double, le temps... | Exemple Python |
|---|---|---|---|
| **O(1)** | Constant | reste identique | `dict[clé]`, `x in set`, `liste[i]` |
| **O(log n)** | Logarithmique | augmente d'une unité | recherche binaire |
| **O(n)** | Linéaire | double | `x in list`, `for x in liste`, `sum()` |
| **O(n log n)** | Quasi-linéaire | un peu plus que double | `sort()`, `sorted()` |
| **O(n²)** | Quadratique | multiplie par 4 | boucle dans une boucle |
| **O(2ⁿ)** | Exponentiel | explose | fibonacci sans cache |

> **En entretien** : passer de O(n²) à O(n) est toujours plus impactant
> que n'importe quelle micro-optimisation.

---
## 4. `list` vs `set` — O(n) vs O(1)

Le test d'appartenance (`x in collection`) est l'un des sujets les plus demandés en entretien.

- **`list`** : Python parcourt du début à la fin jusqu'à trouver → **O(n)**
- **`set`** : Python calcule un hash et va directement au bon endroit → **O(1)**

**Règle pratique** : si tu dois faire des tests d'appartenance répétés,
convertis ta liste en set d'abord.
Le coût de `set(ma_liste)` est O(n) une seule fois, puis chaque test est O(1).

In [ ]:
import timeit

tailles = [1_000, 10_000, 100_000]

print(f"{'Taille':<12} {'in list':>12} {'in set':>12} {'Ratio':>8}")
print("-" * 50)

for n in tailles:
    data_list = list(range(n))
    data_set  = set(range(n))
    cible = n - 1   # pire cas : tout à la fin de la liste

    t_list = timeit.timeit(lambda: cible in data_list, number=1000)
    t_set  = timeit.timeit(lambda: cible in data_set,  number=1000)

    print(f"{n:<12} {t_list:>10.4f}s {t_set:>10.6f}s {t_list/t_set:>7.0f}x")

print()
print("→ Plus n est grand, plus l'écart se creuse — c'est ça Big O")

---
## 5. Listes vs Générateurs

| | Liste `[x for x in ...]` | Générateur `(x for x in ...)` |
|---|---|---|
| Stockage | Tous les éléments en RAM | Un seul élément à la fois |
| Mémoire | O(n) | O(1) |
| Création | Lente (tout calculé) | Instantanée |
| Parcourable | Plusieurs fois | **Une seule fois** |

### Quand utiliser un générateur ?
- Tu ne parcours les données **qu'une seule fois**
- Les données sont **volumineuses** (fichiers, flux, millions d'éléments)
- Tu n'as pas besoin d'accès par index

### Le mot-clé `yield`
Une fonction avec `yield` **se met en pause** et reprend au `next()` suivant.
Elle ne repart pas de zéro — elle reprend exactement là où elle s'est arrêtée.

| Mot-clé | Comportement |
|---|---|
| `return` | Sort de la fonction, fonction **terminée** |
| `yield` | **Pause**, retourne une valeur, **reprend** au prochain `next()` |

In [ ]:
import tracemalloc

N = 1_000_000

# ── Liste : tout calculé et stocké d'un coup ──
tracemalloc.start()
liste = [i**2 for i in range(N)]
_, pic_liste = tracemalloc.get_traced_memory()
tracemalloc.stop()

# ── Générateur : rien stocké ──
tracemalloc.start()
gen = (i**2 for i in range(N))
_, pic_gen = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Liste      : {pic_liste / 1_000_000:.1f} MB")
print(f"Générateur : {pic_gen} bytes")
print()
# sum() accepte un générateur directement
total = sum(i**2 for i in range(N))
print(f"Somme des carrés : {total}  ← calculé sans liste intermédiaire")

In [ ]:
# ── Fonction génératrice avec yield ──

def carres(n):
    for i in range(n):
        yield i ** 2    # pause ici, reprend au next()

# S'utilise comme une liste dans un for
for c in carres(6):
    print(c, end=' ')
print()

# Démonstration de la pause / reprise
g = carres(4)
print(next(g))   # 0  — calcule et pause
print(next(g))   # 1  — reprend, calcule, pause
print(next(g))   # 4
print(next(g))   # 9
# print(next(g)) # → StopIteration : épuisé

---
## 6. `deque` — quand `list.pop(0)` est trop lent

Une liste Python est un tableau de cases contiguës en mémoire.
`pop(0)` supprime la première case → Python décale **toutes** les autres → **O(n)**.

`collections.deque` est conçue pour les opérations aux deux extrémités :
`appendleft` / `popleft` à gauche, `append` / `pop` à droite — tous en **O(1)**.

```python
deque(iterable=None, maxlen=None)
  maxlen : si défini, supprime de l'autre côté automatiquement quand plein
```

In [ ]:
import timeit
from collections import deque

N = 10_000

t_list = timeit.timeit(
    stmt='data.pop(0)',
    setup=f'data = list(range({N}))',
    number=N
)

t_deque = timeit.timeit(
    stmt='data.popleft()',
    setup=f'from collections import deque; data = deque(range({N}))',
    number=N
)

print(f"list.pop(0)     : {t_list:.4f}s  — O(n)")
print(f"deque.popleft() : {t_deque:.4f}s  — O(1)")
print(f"→ deque est {t_list/t_deque:.0f}x plus rapide ici")
print()

# Cas pratique : file d'attente
file = deque()
file.append("Alice")
file.append("Bob")
file.append("Charlie")
premier = file.popleft()   # O(1)
print(f"Premier servi : {premier}")
print(f"File restante : {list(file)}")

---
## 7. Concaténation de chaînes — le piège classique

Les strings Python sont **immutables** — on ne peut pas les modifier.
Chaque `+=` crée une **nouvelle chaîne**, copie l'ancienne dedans, jette l'ancienne.

Sur 1000 itérations : copie de 1 caractère, puis 2, puis 3... = **O(n²)**.

`"".join(liste)` mesure la taille totale, alloue **une seule fois**, copie en une passe → **O(n)**.

```
séparateur.join(itérable_de_strings)
  séparateur : inséré entre chaque élément
  ""  → rien entre les éléments
  " " → un espace
  ", " → virgule-espace
```

In [ ]:
import timeit

mots = [f'mot_{i}' for i in range(1000)]

def avec_plus(mots):
    resultat = ""
    for mot in mots:
        resultat += mot + " "   # nouvelle chaîne à chaque tour — O(n²)
    return resultat

def avec_join(mots):
    return " ".join(mots)       # une seule allocation — O(n)

t1 = timeit.timeit(lambda: avec_plus(mots),  number=2000)
t2 = timeit.timeit(lambda: avec_join(mots),  number=2000)

print(f"+=     : {t1:.4f}s")
print(f"join() : {t2:.4f}s")
print(f"→ join() est {t1/t2:.0f}x plus rapide")
print()

# Exemples de séparateurs
mots_courts = ["Python", "est", "rapide"]
print(" ".join(mots_courts))      # Python est rapide
print(", ".join(mots_courts))     # Python, est, rapide
print(" | ".join(mots_courts))    # Python | est | rapide

---
## Récapitulatif

| Situation | Solution | Complexité |
|---|---|---|
| Mesurer le temps | `timeit.timeit(lambda: ..., number=N)` | — |
| Mesurer la mémoire | `tracemalloc` | — |
| Test `x in collection` répété | `set` ou `dict` | O(1) vs O(n) |
| Gros volume, une seule passe | Générateur `(...)` | O(1) mémoire |
| Ajouter/supprimer en début | `collections.deque` | O(1) vs O(n) |
| Assembler des chaînes en boucle | `"".join(...)` | O(n) vs O(n²) |